In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import duckdb

# Project root
PROJECT_ROOT = Path.cwd().parent

# Raw data
STATIONS_PATH = PROJECT_ROOT / "data" / "raw" / "stations.parquet"
DAILY_PATH = PROJECT_ROOT / "data" / "raw" / "daily.parquet"

# Intermediate / processed data
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# DuckDB connection
con = duckdb.connect()

print("Project root:", PROJECT_ROOT)
print("Daily dataset:", DAILY_PATH)

Project root: D:\01_PROJECTS\DS_AI\flood-prediction-india
Daily dataset: D:\01_PROJECTS\DS_AI\flood-prediction-india\data\raw\daily.parquet


In [2]:
FINAL_STATIONS_PATH = (
    INTERIM_DIR / "final_station_list.parquet"
)

final_stations = pd.read_parquet(
    FINAL_STATIONS_PATH
)

print("Final stations:", len(final_stations))
print(final_stations.shape)

Final stations: 703
(703, 17)


In [3]:
final_stations.head()

,station_code,name,lat,lon,type,nearest_town,warning_level,danger_level,frl,mwl,highest_flow_level,hhs_days,mps_days,sentinel_rate,median_danger_ratio,p99_danger_ratio,overlap_days
0,001-CDBNG,Shimoga,13.926944,75.585000,Base,NaN,566.7,567.25,NaN,NaN,570.18,12959,12638,0.0,0.985203,0.997113,12399
1,001-CDJPR,Dhareri,23.147500,75.508333,Base,,478.5,479.50,NaN,NaN,482.62,12483,14234,0.0,0.972614,0.983106,12032
2,001-HGDDDN,Badrinath,30.748333,79.495833,Base,NaN,3112.0,3113.00,NaN,NaN,3112.70,7361,7877,0.0,0.999426,0.999598,7071
3,001-LBDJPG,Mathanguri,26.782778,90.956667,Level,NaN,98.1,99.10,NaN,NaN,100.28,5086,4489,0.0,0.936245,0.964103,4487
4,001-LYDAGRA,AGRA (J.B.),27.203889,78.035000,Level,Agra,151.4,152.40,NaN,NaN,154.76,12787,2746,0.0,0.962861,0.988439,2741


In [4]:
con.register(
    "final_stations_df",
    final_stations[
        [
            "station_code",
            "name",
            "lat",
            "lon",
            "warning_level",
            "danger_level"
        ]
    ]
)

print("Station list registered with DuckDB.")

Station list registered with DuckDB.


In [5]:
coverage = con.execute(f"""
    SELECT
        datatype_code,
        COUNT(*) AS rows,
        COUNT(DISTINCT station_code) AS stations,
        MIN(date_ist) AS first_date,
        MAX(date_ist) AS last_date
    FROM read_parquet('{DAILY_PATH}')
    WHERE datatype_code IN ('HHS', 'MPS')
      AND station_code IN (
          SELECT station_code
          FROM final_stations_df
      )
    GROUP BY datatype_code
    ORDER BY datatype_code
""").fetchdf()

coverage

,datatype_code,rows,stations,first_date,last_date
0,HHS,5205243,703,1919-06-16,2026-09-11
1,MPS,4642956,703,1960-05-01,2026-09-11


In [6]:
daily_base = con.execute(f"""
    SELECT
        d.station_code,
        d.date_ist AS date,

        MAX(CASE
            WHEN d.datatype_code = 'HHS'
            THEN d.mean
        END) AS water_level,

        MAX(CASE
            WHEN d.datatype_code = 'MPS'
            THEN d.sum
        END) AS rainfall

    FROM read_parquet('{DAILY_PATH}') d

    INNER JOIN final_stations_df s
        ON d.station_code = s.station_code

    WHERE d.datatype_code IN ('HHS', 'MPS')

    GROUP BY
        d.station_code,
        d.date_ist

    ORDER BY
        d.station_code,
        d.date_ist
""").fetchdf()

print(daily_base.shape)
daily_base.head()

(6302319, 4)


,station_code,date,water_level,rainfall
0,001-CDBNG,1970-01-01,NaN,0.0
1,001-CDBNG,1988-06-01,NaN,0.0
2,001-CDBNG,1988-06-02,NaN,0.0
3,001-CDBNG,1988-06-03,NaN,0.0
4,001-CDBNG,1988-06-04,NaN,0.0


In [7]:
daily_base.info()

<class 'pandas.DataFrame'>
RangeIndex: 6302319 entries, 0 to 6302318
Data columns (total 4 columns):
 #   Column        Dtype         
---  ------        -----         
 0   station_code  str           
 1   date          datetime64[us]
 2   water_level   float64       
 3   rainfall      float64       
dtypes: datetime64[us](1), float64(2), str(1)
memory usage: 254.7 MB


In [8]:
daily_base.isna().sum()

station_code          0
date                  0
water_level     1097076
rainfall        1659363
dtype: int64

In [9]:
daily_base.describe()

,date,water_level,rainfall
count,6302319,5.205243e+06,4.642956e+06
mean,2006-11-18 18:47:05.649035,2.302226e+02,-5.639857e+01
min,1919-06-16 00:00:00,-9.990000e+03,-2.397600e+04
25%,1995-10-13 00:00:00,6.341333e+01,0.000000e+00
50%,2010-03-12 00:00:00,1.181800e+02,0.000000e+00
75%,2020-05-11 00:00:00,2.697500e+02,0.000000e+00
max,2026-09-11 00:00:00,6.635595e+05,2.082200e+03
std,NaN,7.276079e+02,1.172804e+03


In [10]:
print("Minimum date:", daily_base["date"].min())
print("Maximum date:", daily_base["date"].max())

Minimum date: 1919-06-16 00:00:00
Maximum date: 2026-09-11 00:00:00


In [11]:
daily_base.groupby("station_code")["date"].agg(
    ["min", "max", "count"]
).describe()

,min,max,count
count,703,703,703.000000
mean,1985-10-07 07:06:03.584637,2026-09-04 20:24:55.305832,8964.891892
min,1919-06-16 00:00:00,2022-10-11 00:00:00,1177.000000
25%,1970-01-01 00:00:00,2026-09-11 00:00:00,2912.500000
50%,1979-06-01 00:00:00,2026-09-11 00:00:00,7253.000000
75%,2001-07-01 12:00:00,2026-09-11 00:00:00,14236.000000
max,2023-06-01 00:00:00,2026-09-11 00:00:00,25806.000000
std,NaN,NaN,6315.898051


In [12]:
print("Water level:")
print(daily_base["water_level"].describe())

print("\nRainfall:")
print(daily_base["rainfall"].describe())

Water level:
count    5.205243e+06
mean     2.302226e+02
std      7.276079e+02
min     -9.990000e+03
25%      6.341333e+01
50%      1.181800e+02
75%      2.697500e+02
max      6.635595e+05
Name: water_level, dtype: float64

Rainfall:
count    4.642956e+06
mean    -5.639857e+01
std      1.172804e+03
min     -2.397600e+04
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.082200e+03
Name: rainfall, dtype: float64


In [13]:
print(
    "Negative water levels:",
    (daily_base["water_level"] < 0).sum()
)

print(
    "Negative rainfall:",
    (daily_base["rainfall"] < 0).sum()
)

Negative water levels: 711
Negative rainfall: 20824


In [14]:
negative_values = con.execute(f"""
    SELECT
        datatype_code,
        COUNT(*) AS negative_rows,
        MIN(mean) AS minimum_mean,
        MAX(mean) AS maximum_negative_mean
    FROM read_parquet('{DAILY_PATH}')
    WHERE station_code IN (
        SELECT station_code
        FROM final_stations_df
    )
    AND datatype_code IN ('HHS', 'MPS')
    AND mean < 0
    GROUP BY datatype_code
    ORDER BY datatype_code
""").fetchdf()

negative_values

,datatype_code,negative_rows,minimum_mean,maximum_negative_mean
0,HHS,711,-9990.0,-0.000400
1,MPS,20824,-999.0,-0.003333


In [15]:
negative_common = con.execute(f"""
    SELECT
        datatype_code,
        mean,
        COUNT(*) AS occurrences
    FROM read_parquet('{DAILY_PATH}')
    WHERE station_code IN (
        SELECT station_code
        FROM final_stations_df
    )
    AND datatype_code IN ('HHS', 'MPS')
    AND mean < 0
    GROUP BY datatype_code, mean
    ORDER BY datatype_code, occurrences DESC
    LIMIT 50
""").fetchdf()

negative_common

,datatype_code,mean,occurrences
0,HHS,-999.000000,90
1,HHS,-0.180000,31
2,HHS,-0.170000,26
3,HHS,-0.200000,19
4,HHS,-0.190000,15
5,HHS,-0.670000,11
6,HHS,-0.150000,9
7,HHS,-0.680000,8
8,HHS,-0.160000,8
9,HHS,-0.240000,7


In [17]:
negative_rainfall = con.execute(f"""
    SELECT
        station_code,
        date_ist AS date,
        min,
        mean,
        max,
        sum,
        n_obs
    FROM read_parquet('{DAILY_PATH}')
    WHERE datatype_code = 'MPS'
      AND station_code IN (
          SELECT station_code
          FROM final_stations_df
      )
      AND sum < 0
    ORDER BY sum
    LIMIT 50
""").fetchdf()

negative_rainfall

,station_code,date,min,mean,max,sum,n_obs
0,013-CDJAPR,1983-09-18,-999.0,-999.00,-999.0,-23976.0,24
1,013-CDJAPR,1987-07-31,-999.0,-999.00,-999.0,-23976.0,24
2,013-CDJAPR,1986-12-17,-999.0,-999.00,-999.0,-23976.0,24
3,013-CDJAPR,1994-08-07,-999.0,-959.04,0.0,-23976.0,25
4,013-CDJAPR,1994-07-11,-999.0,-959.04,0.0,-23976.0,25
5,013-CDJAPR,1994-07-29,-999.0,-959.04,0.0,-23976.0,25
6,013-CDJAPR,1994-08-08,-999.0,-959.04,0.0,-23976.0,25
7,013-CDJAPR,1989-03-16,-999.0,-999.00,-999.0,-23976.0,24
8,013-CDJAPR,1994-08-10,-999.0,-959.04,0.0,-23976.0,25
9,013-CDJAPR,1994-08-01,-999.0,-959.04,0.0,-23976.0,25


In [18]:
extreme_hhs = con.execute(f"""
    SELECT
        station_code,
        date_ist AS date,
        min,
        mean,
        max,
        n_obs
    FROM read_parquet('{DAILY_PATH}')
    WHERE datatype_code = 'HHS'
      AND station_code IN (
          SELECT station_code
          FROM final_stations_df
      )
      AND mean > 10000
    ORDER BY mean DESC
    LIMIT 50
""").fetchdf()

extreme_hhs

,station_code,date,min,mean,max,n_obs
0,017-MGD2LKN,1988-08-05,199.260,663559.513333,1990280.00,3
1,017-MGD2LKN,1969-05-13,195.255,651246.841667,1953350.00,3
2,002-MGD2LKN,1989-07-24,160.687,534673.124667,1603698.00,3
3,011-DDASL,1996-02-15,423687.000,423687.000000,423687.00,1
4,011-DDASL,2004-02-03,422118.000,422118.000000,422118.00,1
5,011-DDASL,2013-07-17,419283.000,419283.000000,419283.00,1
6,027-UYDDEL,2020-02-02,1202.310,401478.206667,1202030.00,3
7,019-CDJAPR,1995-03-15,386.780,129183.853333,386778.00,3
8,019-CDJAPR,1994-02-05,386.880,129053.920000,386388.00,3
9,019-CDJAPR,2000-02-24,386.745,129049.163333,386374.00,3


In [19]:
con.register(
    "final_stations_df",
    final_stations[
        [
            "station_code",
            "name",
            "lat",
            "lon",
            "warning_level",
            "danger_level"
        ]
    ]
)

In [20]:
clean_daily = con.execute(f"""
    SELECT
        d.station_code,
        d.date_ist AS date,

        CASE
            WHEN d.datatype_code = 'HHS'
                 AND d.mean <= -900
                THEN NULL

            WHEN d.datatype_code = 'HHS'
                 AND d.mean > 5 * s.danger_level
                THEN NULL

            WHEN d.datatype_code = 'HHS'
                THEN d.mean

            ELSE NULL
        END AS water_level,

        CASE
            WHEN d.datatype_code = 'MPS'
                 AND d.sum < 0
                THEN NULL

            WHEN d.datatype_code = 'MPS'
                THEN d.sum

            ELSE NULL
        END AS rainfall,

        s.warning_level,
        s.danger_level,
        s.lat,
        s.lon

    FROM read_parquet('{DAILY_PATH}') d

    INNER JOIN final_stations_df s
        ON d.station_code = s.station_code

    WHERE d.datatype_code IN ('HHS', 'MPS')
""").fetchdf()

print(clean_daily.shape)

(9848199, 8)


In [21]:
clean_daily = (
    clean_daily
    .groupby(
        [
            "station_code",
            "date",
            "warning_level",
            "danger_level",
            "lat",
            "lon"
        ],
        as_index=False
    )
    .agg(
        water_level=("water_level", "max"),
        rainfall=("rainfall", "max")
    )
    .sort_values(
        ["station_code", "date"]
    )
)

print(clean_daily.shape)
clean_daily.head()

(6171953, 8)


,station_code,date,warning_level,danger_level,lat,lon,water_level,rainfall
0,001-CDBNG,1970-01-01,566.7,567.25,13.926944,75.585,NaN,0.0
1,001-CDBNG,1988-06-01,566.7,567.25,13.926944,75.585,NaN,0.0
2,001-CDBNG,1988-06-02,566.7,567.25,13.926944,75.585,NaN,0.0
3,001-CDBNG,1988-06-03,566.7,567.25,13.926944,75.585,NaN,0.0
4,001-CDBNG,1988-06-04,566.7,567.25,13.926944,75.585,NaN,0.0


In [22]:
print("Water level:")
print(clean_daily["water_level"].describe())

print("\nRainfall:")
print(clean_daily["rainfall"].describe())

Water level:
count    5.125866e+06
mean     2.296584e+02
std      3.045008e+02
min     -8.788250e+02
25%      6.386333e+01
50%      1.181600e+02
75%      2.703117e+02
max      3.946353e+03
Name: water_level, dtype: float64

Rainfall:
count    4.504508e+06
mean     4.174697e+00
std      1.580168e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.082200e+03
Name: rainfall, dtype: float64


In [23]:
print(
    "Invalid extreme water levels remaining:",
    (clean_daily["water_level"] > 5 * clean_daily["danger_level"]).sum()
)

print(
    "Negative rainfall remaining:",
    (clean_daily["rainfall"] < 0).sum()
)

print(
    "HHS sentinel values remaining:",
    (clean_daily["water_level"] <= -900).sum()
)

Invalid extreme water levels remaining: 0
Negative rainfall remaining: 0
HHS sentinel values remaining: 0


In [24]:
missing_summary = clean_daily[
    ["water_level", "rainfall"]
].isna().sum()

missing_summary

water_level    1046087
rainfall       1667445
dtype: int64

In [25]:
print(
    "Total rows:",
    len(clean_daily)
)

print(
    "Rows with both water level and rainfall:",
    clean_daily[
        clean_daily["water_level"].notna()
        & clean_daily["rainfall"].notna()
    ].shape[0]
)

Total rows: 6171953
Rows with both water level and rainfall: 3469292


In [26]:
negative_hhs_check = clean_daily[
    clean_daily["water_level"] < 0
]["water_level"]

print("Negative HHS observations:", len(negative_hhs_check))
print(negative_hhs_check.describe())

Negative HHS observations: 592
count    592.000000
mean     -68.097406
std      215.054905
min     -878.825000
25%       -0.697500
50%       -0.503333
75%       -0.180000
max       -0.000400
Name: water_level, dtype: float64


In [27]:
print(
    "HHS < -100:",
    (clean_daily["water_level"] < -100).sum()
)

print(
    "HHS < -500:",
    (clean_daily["water_level"] < -500).sum()
)

HHS < -100: 58
HHS < -500: 46


In [28]:
CLEAN_DAILY_PATH = (
    INTERIM_DIR / "clean_daily.parquet"
)

clean_daily.to_parquet(
    CLEAN_DAILY_PATH,
    index=False
)

print(f"Saved: {CLEAN_DAILY_PATH}")

Saved: D:\01_PROJECTS\DS_AI\flood-prediction-india\data\interim\clean_daily.parquet


In [29]:
clean_daily.loc[
    clean_daily["water_level"] < -100,
    "water_level"
] = np.nan

In [30]:
print(
    "HHS < -100 remaining:",
    (clean_daily["water_level"] < -100).sum()
)

print(
    "Negative HHS remaining:",
    (clean_daily["water_level"] < 0).sum()
)

HHS < -100 remaining: 0
Negative HHS remaining: 534


In [31]:
clean_daily.to_parquet(
    CLEAN_DAILY_PATH,
    index=False
)

print(f"Updated: {CLEAN_DAILY_PATH}")

Updated: D:\01_PROJECTS\DS_AI\flood-prediction-india\data\interim\clean_daily.parquet


In [32]:
con.register("clean_daily_df", clean_daily)

In [33]:
calendar_daily = con.execute("""
    SELECT
        s.station_code,
        d.date
    FROM (
        SELECT
            station_code,
            MIN(date) AS start_date,
            MAX(date) AS end_date
        FROM clean_daily_df
        GROUP BY station_code
    ) s
    CROSS JOIN LATERAL generate_series(
        s.start_date,
        s.end_date,
        INTERVAL 1 DAY
    ) AS d(date)
""").fetchdf()

print(calendar_daily.shape)
calendar_daily.head()

(10339811, 2)


,station_code,date
0,008-HGDDDN,2026-09-11
1,021-DDASL,2026-09-11
2,004-HGDDDN,1989-12-27
3,009-MGD2LKN,2026-09-11
4,009-MGD3VNS,2026-09-11


In [34]:
calendar_daily = calendar_daily.merge(
    clean_daily,
    on=["station_code", "date"],
    how="left"
)

In [35]:
station_meta = final_stations[
    [
        "station_code",
        "name",
        "lat",
        "lon",
        "warning_level",
        "danger_level"
    ]
].drop_duplicates("station_code")

In [36]:
calendar_daily = calendar_daily.drop(
    columns=[
        "name",
        "lat",
        "lon",
        "warning_level",
        "danger_level"
    ],
    errors="ignore"
).merge(
    station_meta,
    on="station_code",
    how="left"
)

In [37]:
calendar_daily = calendar_daily[
    [
        "station_code",
        "date",
        "name",
        "lat",
        "lon",
        "warning_level",
        "danger_level",
        "water_level",
        "rainfall"
    ]
].sort_values(
    ["station_code", "date"]
).reset_index(drop=True)

In [38]:
print(calendar_daily.shape)
calendar_daily.head(10)

(10339811, 9)


,station_code,date,name,lat,lon,warning_level,danger_level,water_level,rainfall
0,001-CDBNG,1970-01-01,Shimoga,13.926944,75.585,566.7,567.25,NaN,0.0
1,001-CDBNG,1970-01-02,Shimoga,13.926944,75.585,566.7,567.25,NaN,NaN
2,001-CDBNG,1970-01-03,Shimoga,13.926944,75.585,566.7,567.25,NaN,NaN
3,001-CDBNG,1970-01-04,Shimoga,13.926944,75.585,566.7,567.25,NaN,NaN
4,001-CDBNG,1970-01-05,Shimoga,13.926944,75.585,566.7,567.25,NaN,NaN
5,001-CDBNG,1970-01-06,Shimoga,13.926944,75.585,566.7,567.25,NaN,NaN
6,001-CDBNG,1970-01-07,Shimoga,13.926944,75.585,566.7,567.25,NaN,NaN
7,001-CDBNG,1970-01-08,Shimoga,13.926944,75.585,566.7,567.25,NaN,NaN
8,001-CDBNG,1970-01-09,Shimoga,13.926944,75.585,566.7,567.25,NaN,NaN
9,001-CDBNG,1970-01-10,Shimoga,13.926944,75.585,566.7,567.25,NaN,NaN


In [39]:
date_check = (
    calendar_daily
    .groupby("station_code")["date"]
    .diff()
    .dropna()
)

print(date_check.value_counts().head())

date
1 days    10339122
Name: count, dtype: int64


In [40]:
print(
    "Gaps > 1 day:",
    (date_check > pd.Timedelta(days=1)).sum()
)

Gaps > 1 day: 0


In [42]:
CALENDAR_PATH = (
    INTERIM_DIR / "calendar_daily.parquet"
)

calendar_daily.to_parquet(
    CALENDAR_PATH,
    index=False
)

print(f"Saved: {CALENDAR_PATH}")

Saved: D:\01_PROJECTS\DS_AI\flood-prediction-india\data\interim\calendar_daily.parquet


In [43]:
calendar_daily = calendar_daily.sort_values(
    ["station_code", "date"]
).reset_index(drop=True)

In [44]:
grouped = calendar_daily.groupby(
    "station_code",
    group_keys=False
)

calendar_daily["water_level_lag_1"] = (
    grouped["water_level"].shift(1)
)

calendar_daily["water_level_lag_2"] = (
    grouped["water_level"].shift(2)
)

calendar_daily["water_level_lag_3"] = (
    grouped["water_level"].shift(3)
)

In [45]:
calendar_daily[
    [
        "station_code",
        "date",
        "water_level",
        "water_level_lag_1",
        "water_level_lag_2",
        "water_level_lag_3"
    ]
].head(10)

,station_code,date,water_level,water_level_lag_1,water_level_lag_2,water_level_lag_3
0,001-CDBNG,1970-01-01,NaN,NaN,NaN,NaN
1,001-CDBNG,1970-01-02,NaN,NaN,NaN,NaN
2,001-CDBNG,1970-01-03,NaN,NaN,NaN,NaN
3,001-CDBNG,1970-01-04,NaN,NaN,NaN,NaN
4,001-CDBNG,1970-01-05,NaN,NaN,NaN,NaN
5,001-CDBNG,1970-01-06,NaN,NaN,NaN,NaN
6,001-CDBNG,1970-01-07,NaN,NaN,NaN,NaN
7,001-CDBNG,1970-01-08,NaN,NaN,NaN,NaN
8,001-CDBNG,1970-01-09,NaN,NaN,NaN,NaN
9,001-CDBNG,1970-01-10,NaN,NaN,NaN,NaN


In [46]:
calendar_daily["level_change_1d"] = (
    calendar_daily["water_level"]
    - calendar_daily["water_level_lag_1"]
)

calendar_daily["level_change_3d"] = (
    calendar_daily["water_level"]
    - calendar_daily["water_level_lag_3"]
)

In [47]:
calendar_daily["rain_1d"] = (
    calendar_daily["rainfall"]
)

In [48]:
calendar_daily["rain_3d"] = (
    calendar_daily
    .groupby("station_code")["rainfall"]
    .transform(
        lambda x: x.rolling(
            window=3,
            min_periods=3
        ).sum()
    )
)

In [49]:
calendar_daily["rain_7d"] = (
    calendar_daily
    .groupby("station_code")["rainfall"]
    .transform(
        lambda x: x.rolling(
            window=7,
            min_periods=7
        ).sum()
    )
)

In [50]:
calendar_daily["next_day_water_level"] = (
    calendar_daily
    .groupby("station_code")["water_level"]
    .shift(-1)
)

In [51]:
calendar_daily["flood_next_day"] = (
    calendar_daily["next_day_water_level"]
    >= calendar_daily["danger_level"]
).astype("Int8")

In [52]:
calendar_daily["flood_next_day"] = np.where(
    calendar_daily["next_day_water_level"].isna(),
    np.nan,
    (
        calendar_daily["next_day_water_level"]
        >= calendar_daily["danger_level"]
    ).astype(int)
)

In [53]:
calendar_daily["flood_next_day"].value_counts(
    dropna=False
)

flood_next_day
NaN    5214533
0.0    5029475
1.0      95803
Name: count, dtype: int64

In [54]:
print(
    calendar_daily["flood_next_day"].mean()
)

0.0186922543518615


In [55]:
print(
    "Flood days:",
    (calendar_daily["flood_next_day"] == 1).sum()
)

print(
    "Non-flood days:",
    (calendar_daily["flood_next_day"] == 0).sum()
)

print(
    "Unknown target:",
    calendar_daily["flood_next_day"].isna().sum()
)

Flood days: 95803
Non-flood days: 5029475
Unknown target: 5214533


In [56]:
feature_columns = [
    "water_level",
    "water_level_lag_1",
    "water_level_lag_2",
    "water_level_lag_3",
    "level_change_1d",
    "level_change_3d",
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "danger_level",
    "warning_level",
    "lat",
    "lon",
    "flood_next_day"
]

calendar_daily[feature_columns].describe()

,water_level,water_level_lag_1,water_level_lag_2,water_level_lag_3,level_change_1d,level_change_3d,rain_1d,rain_3d,rain_7d,danger_level,warning_level,lat,lon,flood_next_day
count,5.125808e+06,5.125121e+06,5.124436e+06,5.123751e+06,5.090855e+06,5.070774e+06,4.504508e+06,4.440547e+06,4.349331e+06,1.033981e+07,1.033981e+07,1.033981e+07,1.033981e+07,5.125278e+06
mean,2.296688e+02,2.296644e+02,2.296603e+02,2.296561e+02,6.132562e-03,1.694053e-02,4.174697e+00,1.252663e+01,2.932117e+01,2.560715e+02,2.550033e+02,2.433906e+01,8.168847e+01,1.869225e-02
std,3.044860e+02,3.044807e+02,3.044759e+02,3.044711e+02,2.374479e+01,2.490361e+01,1.580168e+01,3.481635e+01,6.489858e+01,3.586917e+02,3.586713e+02,5.143359e+00,5.247020e+00,1.354358e-01
min,-4.886833e+01,-4.886833e+01,-4.886833e+01,-4.886833e+01,-2.959647e+03,-2.959670e+03,0.000000e+00,0.000000e+00,0.000000e+00,9.200000e+00,8.600000e+00,8.714444e+00,7.018083e+01,0.000000e+00
25%,6.386500e+01,6.386417e+01,6.386333e+01,6.386333e+01,-3.666667e-02,-8.333333e-02,0.000000e+00,0.000000e+00,0.000000e+00,6.275000e+01,6.175000e+01,2.361806e+01,7.774139e+01,0.000000e+00
50%,1.181600e+02,1.181600e+02,1.181600e+02,1.181600e+02,-4.583333e-03,-1.000000e-02,0.000000e+00,0.000000e+00,1.200000e+00,1.275000e+02,1.265000e+02,2.585056e+01,8.031222e+01,0.000000e+00
75%,2.703146e+02,2.703079e+02,2.703046e+02,2.703000e+02,1.000000e-02,3.333333e-02,0.000000e+00,8.500000e+00,3.200000e+01,2.940000e+02,2.930000e+02,2.701611e+01,8.537611e+01,0.000000e+00
max,3.946353e+03,3.946353e+03,3.946353e+03,3.946353e+03,2.959643e+03,2.959700e+03,2.082200e+03,2.872800e+03,5.310300e+03,3.113000e+03,3.112000e+03,3.424669e+01,9.702583e+01,1.000000e+00


In [57]:
calendar_daily[
    [
        "station_code",
        "date",
        "water_level",
        "water_level_lag_1",
        "level_change_1d",
        "rain_1d",
        "rain_3d",
        "rain_7d",
        "danger_level",
        "flood_next_day"
    ]
].head(20)

,station_code,date,water_level,water_level_lag_1,level_change_1d,rain_1d,rain_3d,rain_7d,danger_level,flood_next_day
0,001-CDBNG,1970-01-01,NaN,NaN,NaN,0.0,NaN,NaN,567.25,NaN
1,001-CDBNG,1970-01-02,NaN,NaN,NaN,NaN,NaN,NaN,567.25,NaN
2,001-CDBNG,1970-01-03,NaN,NaN,NaN,NaN,NaN,NaN,567.25,NaN
3,001-CDBNG,1970-01-04,NaN,NaN,NaN,NaN,NaN,NaN,567.25,NaN
4,001-CDBNG,1970-01-05,NaN,NaN,NaN,NaN,NaN,NaN,567.25,NaN
5,001-CDBNG,1970-01-06,NaN,NaN,NaN,NaN,NaN,NaN,567.25,NaN
6,001-CDBNG,1970-01-07,NaN,NaN,NaN,NaN,NaN,NaN,567.25,NaN
7,001-CDBNG,1970-01-08,NaN,NaN,NaN,NaN,NaN,NaN,567.25,NaN
8,001-CDBNG,1970-01-09,NaN,NaN,NaN,NaN,NaN,NaN,567.25,NaN
9,001-CDBNG,1970-01-10,NaN,NaN,NaN,NaN,NaN,NaN,567.25,NaN


In [58]:
flood_events = calendar_daily[
    calendar_daily["flood_next_day"] == 1
].copy()

print("Flood-event rows:", len(flood_events))

display(
    flood_events[
        [
            "station_code",
            "date",
            "water_level",
            "next_day_water_level",
            "danger_level",
            "rain_1d",
            "rain_3d",
            "rain_7d",
            "flood_next_day"
        ]
    ].head(20)
)

Flood-event rows: 95803


,station_code,date,water_level,next_day_water_level,danger_level,rain_1d,rain_3d,rain_7d,flood_next_day
15559,001-CDBNG,2012-08-07,567.178148,567.478889,567.25,31.2,34.6,43.8,1.0
16282,001-CDBNG,2014-07-31,566.905417,567.360000,567.25,68.2,106.0,156.8,1.0
16283,001-CDBNG,2014-08-01,567.360000,567.823333,567.25,73.0,172.0,208.8,1.0
17756,001-CDBNG,2018-08-13,566.558333,567.912917,567.25,35.2,54.0,75.0,1.0
17757,001-CDBNG,2018-08-14,567.912917,567.952917,567.25,62.8,116.8,127.4,1.0
18114,001-CDBNG,2019-08-06,567.160417,568.256667,567.25,68.0,84.2,92.6,1.0
18115,001-CDBNG,2019-08-07,568.256667,568.465417,567.25,95.4,176.2,185.4,1.0
18116,001-CDBNG,2019-08-08,568.465417,568.734167,567.25,23.4,186.8,208.8,1.0
18117,001-CDBNG,2019-08-09,568.734167,569.702083,567.25,0.0,118.8,208.8,1.0
18118,001-CDBNG,2019-08-10,569.702083,568.777500,567.25,81.2,104.6,284.2,1.0


In [59]:
flood_events[
    ["water_level", "next_day_water_level", "danger_level"]
].describe()

,water_level,next_day_water_level,danger_level
count,94927.000000,95803.000000,95803.000000
mean,142.370515,145.400212,137.862217
std,182.280480,188.155097,171.357524
min,0.000000,9.210625,9.200000
25%,48.085903,48.217708,47.530000
50%,75.881250,77.348333,74.980000
75%,140.421250,140.700500,139.500000
max,2201.713759,3946.353333,1786.450000


In [60]:
flood_events["station_code"].value_counts().head(20)

station_code
014-UBDDIB       5371
024-MGD4PTN      3434
DWRIS-04         2623
044-HDCHENNAI    2591
038-HDCHENNAI    2259
008-CDJAPR       2104
008-MGD1LKN      2089
017-MGD4PTN      1959
025-MGD4PTN      1866
012-MGD1LKN      1706
048              1565
006-MGD4PTN      1441
010-MGD1LKN      1390
005-LGD3BEH      1379
011-MGD1LKN      1316
018-MGD4PTN      1290
011-MGD4PTN      1287
016-HGDDDN       1281
049              1215
007-MGD5PTN      1198
Name: count, dtype: int64

In [61]:
print(
    "Stations with at least one flood event:",
    flood_events["station_code"].nunique()
)

print(
    "Total selected stations:",
    calendar_daily["station_code"].nunique()
)

Stations with at least one flood event: 589
Total selected stations: 689


In [62]:
flood_events[
    ["station_code", "date", "water_level",
     "next_day_water_level", "danger_level"]
].sort_values(
    "next_day_water_level",
    ascending=False
).head(30)

,station_code,date,water_level,next_day_water_level,danger_level
7476333,031-MGD1LKN,2019-05-10,986.710000,3946.353333,988.250
271540,001-UYDDEL,2007-10-21,1058.380000,3558.367500,1060.500
852983,003-UYDDEL,2006-03-10,924.620000,3005.490000,930.500
1162206,005-HGDDDN,1982-07-31,824.461481,2886.226667,624.700
1161905,005-HGDDDN,1981-10-03,614.203333,2862.850741,624.700
1135936,004-UYDDEL,2006-03-07,886.250000,2861.285000,902.500
7052062,028-MGD1LKN,2014-08-04,605.355556,2825.352963,607.300
1158144,005-HGDDDN,1971-06-17,1381.429630,2201.713759,624.700
6819936,026CHEJMU,2009-02-15,1782.656667,2200.780000,1786.450
7359728,030-MGD1LKN,2019-07-03,1549.588750,2129.384583,1551.100


In [63]:
flood_events["next_day_level_danger_ratio"] = (
    flood_events["next_day_water_level"]
    / flood_events["danger_level"]
)

flood_events["next_day_level_danger_ratio"].describe()

count    95803.000000
mean         1.037378
std          0.187172
min          1.000000
25%          1.003188
50%          1.008865
75%          1.023374
max          4.992863
Name: next_day_level_danger_ratio, dtype: float64

In [64]:
flood_events[
    ["station_code", "date",
     "next_day_water_level",
     "danger_level",
     "next_day_level_danger_ratio"]
].sort_values(
    "next_day_level_danger_ratio",
    ascending=False
).head(30)

,station_code,date,next_day_water_level,danger_level,next_day_level_danger_ratio
9389896,051-LYDAGRA,2019-07-08,813.836667,163.000,4.992863
9564359,054-TDSURAT,2019-09-20,758.167917,152.000,4.987947
7132148,029-DDASL,1994-07-12,704.479167,141.280,4.986404
9565483,054-TDSURAT,2022-10-18,757.435000,152.000,4.983125
4428194,016-MAHGAND,2014-11-02,1074.405000,216.000,4.974097
8038891,035-SWRDKOCHI,2019-07-27,483.432917,97.200,4.973590
8447917,039LBDJPG,2016-09-11,212.910833,42.850,4.968748
1668060,006-NDBHP,2017-09-15,1754.127083,354.000,4.955161
5259350,019-MGD4PTN,1988-09-16,259.474583,52.370,4.954642
5305826,019-NDBHP,2018-06-29,1653.851250,334.000,4.951650


In [65]:
calendar_daily["station_code"].nunique()

689

In [66]:
missing_stations = final_stations[
    ~final_stations["station_code"].isin(
        calendar_daily["station_code"].unique()
    )
]

print("Stations missing from calendar_daily:", len(missing_stations))

display(
    missing_stations[
        [
            "station_code",
            "name",
            "danger_level",
            "hhs_days",
            "mps_days",
            "overlap_days"
        ]
    ]
)

Stations missing from calendar_daily: 14


,station_code,name,danger_level,hhs_days,mps_days,overlap_days
132,008-DDASL,Tenughat Reservoir,268.834,10289,9542,8624
141,008-MDSIL,ANIPUR,19.000,5439,6477,4373
170,01-11-06-003,Dholai,24.580,9143,8778,8314
194,011-DDASL,Konar Reservoir,427.940,12248,8742,8538
212,012-MBDGHY,Gharmura,28.050,4074,3958,3953
228,013-ERDBWN,Swampatna,311.100,3692,19837,3528
285,016-MBDGHY,Dholai,24.580,2475,2480,2381
303,017-MBDGHY,Amraghat,26.960,4077,3983,3980
311,017-TDSURAT,BHUSAWAL,185.320,4347,3777,3738
333,019-LKDHYD,Dameracherla,62.040,3485,9919,3442


In [67]:
suspicious_events = flood_events[
    flood_events["next_day_level_danger_ratio"] >= 3
].copy()

print("Events with ratio >= 3:",
      len(suspicious_events))

print("Stations involved:",
      suspicious_events["station_code"].nunique())

display(
    suspicious_events[
        [
            "station_code",
            "date",
            "water_level",
            "next_day_water_level",
            "danger_level",
            "next_day_level_danger_ratio"
        ]
    ].sort_values(
        "next_day_level_danger_ratio",
        ascending=False
    ).head(50)
)

Events with ratio >= 3: 295
Stations involved: 118


,station_code,date,water_level,next_day_water_level,danger_level,next_day_level_danger_ratio
9389896,051-LYDAGRA,2019-07-08,158.008750,813.836667,163.000,4.992863
9564359,054-TDSURAT,2019-09-20,148.489167,758.167917,152.000,4.987947
7132148,029-DDASL,1994-07-12,137.346250,704.479167,141.280,4.986404
9565483,054-TDSURAT,2022-10-18,146.000000,757.435000,152.000,4.983125
4428194,016-MAHGAND,2014-11-02,209.640000,1074.405000,216.000,4.974097
8038891,035-SWRDKOCHI,2019-07-27,94.399583,483.432917,97.200,4.973590
8447917,039LBDJPG,2016-09-11,41.435000,212.910833,42.850,4.968748
1668060,006-NDBHP,2017-09-15,342.361667,1754.127083,354.000,4.955161
5259350,019-MGD4PTN,1988-09-16,51.549167,259.474583,52.370,4.954642
5305826,019-NDBHP,2018-06-29,322.975000,1653.851250,334.000,4.951650


In [68]:
for threshold in [1.5, 2, 2.5, 3, 4, 4.5]:
    count = (
        flood_events["next_day_level_danger_ratio"] >= threshold
    ).sum()

    print(
        f"Ratio >= {threshold}: {count:,}"
    )

Ratio >= 1.5: 939
Ratio >= 2: 460
Ratio >= 2.5: 346
Ratio >= 3: 295
Ratio >= 4: 108
Ratio >= 4.5: 74


In [70]:
station = suspicious_events.iloc[0]["station_code"]
suspicious_date = suspicious_events.iloc[0]["date"]

display(
    calendar_daily[
        (calendar_daily["station_code"] == station) &
        (
            calendar_daily["date"].between(
                suspicious_date - pd.Timedelta(days=7),
                suspicious_date + pd.Timedelta(days=7)
            )
        )
    ][
        [
            "date",
            "water_level",
            "next_day_water_level",
            "danger_level",
            "rain_1d",
            "rain_3d",
            "rain_7d",
            "flood_next_day"
        ]
    ]
)

,date,water_level,next_day_water_level,danger_level,rain_1d,rain_3d,rain_7d,flood_next_day
223715,2019-07-20,73.640000,73.640000,76.8,3.4,3.4,5.6,0.0
223716,2019-07-21,73.640000,73.640000,76.8,23.8,27.2,29.4,0.0
223717,2019-07-22,73.640000,73.640000,76.8,20.4,47.6,49.8,0.0
223718,2019-07-23,73.640000,73.640000,76.8,40.6,84.8,89.8,0.0
223719,2019-07-24,73.640000,73.640000,76.8,0.0,61.0,88.2,0.0
223720,2019-07-25,73.640000,73.680000,76.8,15.2,55.8,103.4,0.0
223721,2019-07-26,73.680000,74.600000,76.8,20.6,35.8,124.0,0.0
223722,2019-07-27,74.600000,365.254400,76.8,96.0,131.8,216.6,1.0
223723,2019-07-28,365.254400,73.640000,76.8,19.2,135.8,212.0,0.0
223724,2019-07-29,73.640000,73.640000,76.8,59.4,174.6,251.0,0.0


In [71]:
# Make sure the data is sorted
calendar_daily = calendar_daily.sort_values(
    ["station_code", "date"]
).reset_index(drop=True)

# Previous and next day's water levels
calendar_daily["prev_water_level"] = (
    calendar_daily
    .groupby("station_code")["water_level"]
    .shift(1)
)

calendar_daily["next2_water_level"] = (
    calendar_daily
    .groupby("station_code")["water_level"]
    .shift(-2)
)

# Ratio of current water level to danger level
calendar_daily["water_danger_ratio"] = (
    calendar_daily["water_level"]
    / calendar_daily["danger_level"]
)

isolated_spikes = calendar_daily[
    (calendar_daily["water_danger_ratio"] >= 3) &
    (calendar_daily["prev_water_level"] < calendar_daily["danger_level"]) &
    (calendar_daily["next2_water_level"] < calendar_daily["danger_level"])
].copy()

print("Isolated spikes >= 3x danger:", len(isolated_spikes))
print(
    "Stations affected:",
    isolated_spikes["station_code"].nunique()
)

display(
    isolated_spikes[
        [
            "station_code",
            "date",
            "prev_water_level",
            "water_level",
            "next2_water_level",
            "danger_level",
            "water_danger_ratio"
        ]
    ]
    .sort_values("water_danger_ratio", ascending=False)
    .head(30)
)

Isolated spikes >= 3x danger: 211
Stations affected: 109


,station_code,date,prev_water_level,water_level,next2_water_level,danger_level,water_danger_ratio
9389897,051-LYDAGRA,2019-07-09,158.008750,813.836667,159.478750,163.000,4.992863
9564360,054-TDSURAT,2019-09-21,148.489167,758.167917,146.835833,152.000,4.987947
7132149,029-DDASL,1994-07-13,137.346250,704.479167,137.566667,141.280,4.986404
9565484,054-TDSURAT,2022-10-19,146.000000,757.435000,146.359167,152.000,4.983125
4428195,016-MAHGAND,2014-11-03,209.640000,1074.405000,209.626667,216.000,4.974097
8038892,035-SWRDKOCHI,2019-07-28,94.399583,483.432917,94.049167,97.200,4.973590
8447918,039LBDJPG,2016-09-12,41.435000,212.910833,41.362917,42.850,4.968748
1668061,006-NDBHP,2017-09-16,342.361667,1754.127083,342.360833,354.000,4.955161
5259351,019-MGD4PTN,1988-09-17,51.549167,259.474583,49.022083,52.370,4.954642
5305827,019-NDBHP,2018-06-30,322.975000,1653.851250,322.711852,334.000,4.951650


In [72]:
for threshold in [1.5, 2, 2.5, 3, 4]:
    count = (
        isolated_spikes["water_danger_ratio"] >= threshold
    ).sum()

    print(
        f"Isolated spikes >= {threshold}x danger: {count:,}"
    )

Isolated spikes >= 1.5x danger: 211
Isolated spikes >= 2x danger: 211
Isolated spikes >= 2.5x danger: 211
Isolated spikes >= 3x danger: 211
Isolated spikes >= 4x danger: 93


In [73]:
calendar_daily["isolated_spike"] = (
    (calendar_daily["water_danger_ratio"] >= 3) &
    (calendar_daily["prev_water_level"] < calendar_daily["danger_level"]) &
    (calendar_daily["next2_water_level"] < calendar_daily["danger_level"])
)

print(
    "Isolated spikes:",
    calendar_daily["isolated_spike"].sum()
)

Isolated spikes: 211


In [74]:
ml_daily = calendar_daily.copy()

In [75]:
ml_daily.loc[
    ml_daily["isolated_spike"],
    "water_level"
] = np.nan

In [76]:
print(
    "Water levels invalidated:",
    ml_daily["water_level"].isna().sum()
    - calendar_daily["water_level"].isna().sum()
)

Water levels invalidated: 211


In [77]:
display(
    ml_daily[
        ml_daily["isolated_spike"]
    ][
        [
            "station_code",
            "date",
            "prev_water_level",
            "water_level",
            "next2_water_level",
            "danger_level",
            "water_danger_ratio"
        ]
    ].head(20)
)

,station_code,date,prev_water_level,water_level,next2_water_level,danger_level,water_danger_ratio
223723,001-TDSURAT,2019-07-28,74.600000,NaN,73.640000,76.80,4.755917
224135,001-TDSURAT,2020-09-12,71.473333,NaN,71.447083,76.80,4.769466
224148,001-TDSURAT,2020-09-25,69.813750,NaN,71.500833,76.80,4.784147
224426,001-TDSURAT,2021-06-30,70.670000,NaN,70.662609,76.80,4.715951
271541,001-UYDDEL,2007-10-22,1058.380000,NaN,1058.347500,1060.50,3.355368
408731,002-MAHGAND,2015-02-03,133.320000,NaN,133.846667,146.00,3.656484
536036,002-TDSURAT,2019-08-07,21.030833,NaN,23.879167,26.80,3.911723
536083,002-TDSURAT,2019-09-23,20.137500,NaN,20.083333,26.80,3.714863
537370,002-TDSURAT,2023-04-02,19.280000,NaN,19.280000,26.80,3.206965
537862,002-TDSURAT,2024-08-06,20.735000,NaN,20.221667,26.80,3.933909


In [78]:
diagnostic_cols = [
    "prev_water_level",
    "next2_water_level",
    "water_danger_ratio",
    "isolated_spike"
]

ml_daily = ml_daily.drop(columns=diagnostic_cols)

In [79]:
ml_daily = ml_daily.sort_values(
    ["station_code", "date"]
).reset_index(drop=True)

In [80]:
ml_daily["water_level_lag_1"] = (
    ml_daily
    .groupby("station_code")["water_level"]
    .shift(1)
)

ml_daily["water_level_lag_2"] = (
    ml_daily
    .groupby("station_code")["water_level"]
    .shift(2)
)

ml_daily["water_level_lag_3"] = (
    ml_daily
    .groupby("station_code")["water_level"]
    .shift(3)
)

In [81]:
ml_daily["level_change_1d"] = (
    ml_daily["water_level"]
    - ml_daily["water_level_lag_1"]
)

ml_daily["level_change_3d"] = (
    ml_daily["water_level"]
    - ml_daily["water_level_lag_3"]
)

In [82]:
ml_daily["rain_1d"] = ml_daily["rainfall"]

ml_daily["rain_3d"] = (
    ml_daily
    .groupby("station_code")["rainfall"]
    .transform(
        lambda x: x.rolling(
            window=3,
            min_periods=3
        ).sum()
    )
)

ml_daily["rain_7d"] = (
    ml_daily
    .groupby("station_code")["rainfall"]
    .transform(
        lambda x: x.rolling(
            window=7,
            min_periods=7
        ).sum()
    )
)

In [83]:
ml_daily["next_day_water_level"] = (
    ml_daily
    .groupby("station_code")["water_level"]
    .shift(-1)
)

In [84]:
ml_daily["flood_next_day"] = np.where(
    ml_daily["next_day_water_level"].isna(),
    np.nan,
    (
        ml_daily["next_day_water_level"]
        >= ml_daily["danger_level"]
    ).astype(float)
)

In [85]:
print(
    ml_daily["flood_next_day"]
    .value_counts(dropna=False)
)

print()

print(
    "Flood days:",
    (ml_daily["flood_next_day"] == 1).sum()
)

print(
    "Non-flood days:",
    (ml_daily["flood_next_day"] == 0).sum()
)

print(
    "Unknown target:",
    ml_daily["flood_next_day"].isna().sum()
)

print()

known_target = ml_daily["flood_next_day"].notna()

print(
    "Flood rate among known targets:",
    ml_daily.loc[
        known_target,
        "flood_next_day"
    ].mean()
)

flood_next_day
NaN    5214744
0.0    5029475
1.0      95592
Name: count, dtype: int64

Flood days: 95592
Non-flood days: 5029475
Unknown target: 5214744

Flood rate among known targets: 0.01865185372210744


In [86]:
display(
    ml_daily[
        (ml_daily["station_code"] == "001-TDSURAT") &
        (
            ml_daily["date"].between(
                pd.Timestamp("2019-07-25"),
                pd.Timestamp("2019-07-30")
            )
        )
    ][
        [
            "date",
            "water_level",
            "water_level_lag_1",
            "level_change_1d",
            "next_day_water_level",
            "danger_level",
            "rain_1d",
            "rain_3d",
            "rain_7d",
            "flood_next_day"
        ]
    ]
)

,date,water_level,water_level_lag_1,level_change_1d,next_day_water_level,danger_level,rain_1d,rain_3d,rain_7d,flood_next_day
223720,2019-07-25,73.64,73.64,0.00,73.68,76.8,15.2,55.8,103.4,0.0
223721,2019-07-26,73.68,73.64,0.04,74.60,76.8,20.6,35.8,124.0,0.0
223722,2019-07-27,74.60,73.68,0.92,NaN,76.8,96.0,131.8,216.6,NaN
223723,2019-07-28,NaN,74.60,NaN,73.64,76.8,19.2,135.8,212.0,0.0
223724,2019-07-29,73.64,NaN,NaN,73.64,76.8,59.4,174.6,251.0,0.0
223725,2019-07-30,73.64,73.64,0.00,73.64,76.8,34.0,112.6,244.4,0.0


In [87]:
feature_cols = [
    "water_level",
    "water_level_lag_1",
    "water_level_lag_2",
    "water_level_lag_3",
    "level_change_1d",
    "level_change_3d",
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "danger_level",
    "warning_level",
    "lat",
    "lon"
]

target_col = "flood_next_day"

print("Total calendar rows:", len(ml_daily))

print(
    "Known target:",
    ml_daily[target_col].notna().sum()
)

print("\nMissing values among features:")

display(
    ml_daily[feature_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

print("\nMissing percentage:")

display(
    (
        ml_daily[feature_cols]
        .isna()
        .mean()
        .mul(100)
        .sort_values(ascending=False)
        .round(2)
        .to_frame("missing_percent")
    )
)

Total calendar rows: 10339811
Known target: 5125067

Missing values among features:


,missing_count
rain_7d,5990480
rain_3d,5899264
rain_1d,5835303
level_change_3d,5269454
level_change_1d,5249378
water_level_lag_3,5216271
water_level_lag_2,5215586
water_level_lag_1,5214901
water_level,5214214
danger_level,0



Missing percentage:


,missing_percent
rain_7d,57.94
rain_3d,57.05
rain_1d,56.44
level_change_3d,50.96
level_change_1d,50.77
water_level_lag_3,50.45
water_level_lag_2,50.44
water_level_lag_1,50.44
water_level,50.43
danger_level,0.00


In [88]:
known_target = ml_daily[target_col].notna()

print(
    "Rows with known target:",
    known_target.sum()
)

for cols in [
    ["water_level"],
    [
        "water_level",
        "water_level_lag_1",
        "water_level_lag_2",
        "water_level_lag_3"
    ],
    [
        "water_level",
        "water_level_lag_1",
        "water_level_lag_2",
        "water_level_lag_3",
        "rain_1d"
    ],
    [
        "water_level",
        "water_level_lag_1",
        "water_level_lag_2",
        "water_level_lag_3",
        "rain_1d",
        "rain_3d",
        "rain_7d"
    ]
]:
    usable = known_target & ml_daily[cols].notna().all(axis=1)

    print(
        f"{len(cols)} required features -> "
        f"{usable.sum():,} usable rows"
    )

Rows with known target: 5125067
1 required features -> 5,090,433 usable rows
4 required features -> 5,015,880 usable rows
5 required features -> 3,411,986 usable rows
7 required features -> 3,292,158 usable rows


In [89]:
ml_daily["level_danger_ratio"] = (
    ml_daily["water_level"]
    / ml_daily["danger_level"]
)

ml_daily["level_warning_ratio"] = (
    ml_daily["water_level"]
    / ml_daily["warning_level"]
)

ml_daily["rise_danger_ratio_1d"] = (
    ml_daily["level_change_1d"]
    / ml_daily["danger_level"]
)

ml_daily["rise_danger_ratio_3d"] = (
    ml_daily["level_change_3d"]
    / ml_daily["danger_level"]
)

In [90]:
display(
    ml_daily[
        [
            "level_danger_ratio",
            "level_warning_ratio",
            "rise_danger_ratio_1d",
            "rise_danger_ratio_3d"
        ]
    ].describe()
)

,level_danger_ratio,level_warning_ratio,rise_danger_ratio_1d,rise_danger_ratio_3d
count,5.125597e+06,5.125597e+06,5.090433e+06,5.070357e+06
mean,9.384713e-01,9.487015e-01,2.247988e-05,7.053805e-05
std,1.115376e-01,1.097529e-01,2.833257e-02,3.318781e-02
min,-8.229451e-01,-8.420833e-01,-3.964127e+00,-3.966846e+00
25%,9.305455e-01,9.435851e-01,-2.969121e-04,-6.709158e-04
50%,9.690297e-01,9.771465e-01,-1.564129e-05,-5.494505e-05
75%,9.857439e-01,9.908747e-01,5.342832e-05,1.948406e-04
max,4.916695e+00,4.950142e+00,3.932211e+00,3.949459e+00


In [91]:
display(
    ml_daily[
        [
            "level_danger_ratio",
            "level_warning_ratio",
            "rise_danger_ratio_1d",
            "rise_danger_ratio_3d"
        ]
    ].quantile(
        [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

,level_danger_ratio,level_warning_ratio,rise_danger_ratio_1d,rise_danger_ratio_3d
0.01,0.295176,0.296250,-0.010123,-0.021993
0.05,0.813421,0.836421,-0.003019,-0.006727
0.25,0.930545,0.943585,-0.000297,-0.000671
0.50,0.969030,0.977146,-0.000016,-0.000055
0.75,0.985744,0.990875,0.000053,0.000195
0.95,0.996509,1.000388,0.003033,0.007080
0.99,1.007664,1.026588,0.013333,0.028791


In [92]:
for threshold in [1.05, 1.10, 1.25, 1.5, 2, 3, 4]:
    count = (
        ml_daily["level_danger_ratio"] >= threshold
    ).sum()

    print(
        f"Current level >= {threshold:.2f} × danger: {count:,}"
    )

Current level >= 1.05 × danger: 12,448
Current level >= 1.10 × danger: 4,385
Current level >= 1.25 × danger: 1,187
Current level >= 1.50 × danger: 732
Current level >= 2.00 × danger: 251
Current level >= 3.00 × danger: 85
Current level >= 4.00 × danger: 15


In [93]:
display(
    ml_daily[
        ml_daily["level_danger_ratio"] >= 2
    ][
        [
            "station_code",
            "date",
            "water_level",
            "danger_level",
            "level_danger_ratio",
            "level_change_1d",
            "level_change_3d"
        ]
    ]
    .sort_values("level_danger_ratio", ascending=False)
    .head(30)
)

,station_code,date,water_level,danger_level,level_danger_ratio,level_change_1d,level_change_3d
8698650,042-TDSURAT,2021-07-26,1091.506250,222.000,4.916695,872.950833,876.780000
3434895,012-UGDHYD,2014-08-02,1823.750000,375.000,4.863333,1459.000000,1459.000000
3434899,012-UGDHYD,2014-08-06,1823.750000,375.000,4.863333,1459.000000,1459.000000
3434897,012-UGDHYD,2014-08-04,1823.750000,375.000,4.863333,1459.000000,1459.000000
4835795,017-NDBHP,2018-08-05,714.832083,150.000,4.765547,NaN,575.570417
1162207,005-HGDDDN,1982-08-01,2886.226667,624.700,4.620180,2061.765185,2268.313704
5558346,020-NDBHP,2017-07-03,1596.463333,350.000,4.561324,NaN,1253.391061
867335,003NEID-3,2023-10-15,989.266667,220.000,4.496667,795.270833,793.277500
5885522,022-DDASL,2007-08-14,123.946250,27.988,4.428550,100.444511,100.435833
6463345,024MAHGAND,2014-10-13,192.394167,45.000,4.275426,154.844167,154.834167


In [94]:
core_features = [
    "water_level",
    "water_level_lag_1",
    "water_level_lag_2",
    "water_level_lag_3",
    "level_change_1d",
    "level_change_3d",
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "level_danger_ratio",
    "level_warning_ratio",
    "rise_danger_ratio_1d",
    "rise_danger_ratio_3d"
]

context_features = [
    "danger_level",
    "warning_level",
    "lat",
    "lon"
]

target_col = "flood_next_day"

usable_mask = (
    ml_daily[target_col].notna()
    & ml_daily[core_features].notna().all(axis=1)
)

print("Usable ML rows:", usable_mask.sum())
print("Dropped rows:", (~usable_mask).sum())

print(
    "Flood rows:",
    (
        usable_mask
        & (ml_daily[target_col] == 1)
    ).sum()
)

print(
    "Non-flood rows:",
    (
        usable_mask
        & (ml_daily[target_col] == 0)
    ).sum()
)

Usable ML rows: 3292158
Dropped rows: 7047653
Flood rows: 54914
Non-flood rows: 3237244


In [95]:
final_ml_preview = ml_daily.loc[
    usable_mask,
    ["station_code", "date", target_col]
].copy()

print(
    "Stations represented:",
    final_ml_preview["station_code"].nunique()
)

display(
    final_ml_preview["station_code"]
    .value_counts()
    .describe()
)

Stations represented: 689


count      689.00000
mean      4778.16836
std       3742.04216
min        908.00000
25%       2324.00000
50%       3435.00000
75%       5872.00000
max      19852.00000
Name: count, dtype: float64

In [96]:
final_features = core_features + context_features

ml_dataset = ml_daily.loc[
    usable_mask,
    [
        "station_code",
        "date",
        *final_features,
        target_col
    ]
].copy()

print("Shape:", ml_dataset.shape)

display(ml_dataset.head())

Shape: (3292158, 20)


,station_code,date,water_level,water_level_lag_1,water_level_lag_2,water_level_lag_3,level_change_1d,level_change_3d,rain_1d,rain_3d,rain_7d,level_danger_ratio,level_warning_ratio,rise_danger_ratio_1d,rise_danger_ratio_3d,danger_level,warning_level,lat,lon,flood_next_day
6912,001-CDBNG,1988-12-04,557.690000,557.670000,557.688333,557.655000,0.020000,0.035000,0.0,0.0,0.0,0.983147,0.984101,0.000035,0.000062,567.25,566.7,13.926944,75.585,0.0
6913,001-CDBNG,1988-12-05,557.706667,557.690000,557.670000,557.688333,0.016667,0.018333,0.0,0.0,0.0,0.983176,0.984130,0.000029,0.000032,567.25,566.7,13.926944,75.585,0.0
6914,001-CDBNG,1988-12-06,557.696667,557.706667,557.690000,557.670000,-0.010000,0.026667,0.0,0.0,0.0,0.983159,0.984113,-0.000018,0.000047,567.25,566.7,13.926944,75.585,0.0
6915,001-CDBNG,1988-12-07,557.688333,557.696667,557.706667,557.690000,-0.008333,-0.001667,0.0,0.0,0.0,0.983144,0.984098,-0.000015,-0.000003,567.25,566.7,13.926944,75.585,0.0
6916,001-CDBNG,1988-12-08,557.676667,557.688333,557.696667,557.706667,-0.011667,-0.030000,0.0,0.0,0.0,0.983123,0.984077,-0.000021,-0.000053,567.25,566.7,13.926944,75.585,0.0


In [97]:
print("Missing values:")
print(ml_dataset.isna().sum().sum())

print("\nDuplicate station-date rows:")
print(
    ml_dataset.duplicated(
        subset=["station_code", "date"]
    ).sum()
)

print("\nTarget distribution:")
print(
    ml_dataset["flood_next_day"]
    .value_counts()
)

print("\nStations:")
print(
    ml_dataset["station_code"].nunique()
)

Missing values:
0

Duplicate station-date rows:
0

Target distribution:
flood_next_day
0.0    3237244
1.0      54914
Name: count, dtype: int64

Stations:
689


In [98]:
ML_DATASET_PATH = PROCESSED_DIR / "ml_dataset.parquet"

ml_dataset.to_parquet(
    ML_DATASET_PATH,
    index=False
)

print(
    f"Saved ML dataset to:\n{ML_DATASET_PATH}"
)

Saved ML dataset to:
D:\01_PROJECTS\DS_AI\flood-prediction-india\data\processed\ml_dataset.parquet


In [99]:
check = pd.read_parquet(ML_DATASET_PATH)

print("Reloaded shape:", check.shape)
print("Columns:")
print(check.columns.tolist())

print("\nTarget distribution:")
print(check["flood_next_day"].value_counts())

Reloaded shape: (3292158, 20)
Columns:
['station_code', 'date', 'water_level', 'water_level_lag_1', 'water_level_lag_2', 'water_level_lag_3', 'level_change_1d', 'level_change_3d', 'rain_1d', 'rain_3d', 'rain_7d', 'level_danger_ratio', 'level_warning_ratio', 'rise_danger_ratio_1d', 'rise_danger_ratio_3d', 'danger_level', 'warning_level', 'lat', 'lon', 'flood_next_day']

Target distribution:
flood_next_day
0.0    3237244
1.0      54914
Name: count, dtype: int64


In [100]:
print("Overall date range:")
print("Start:", ml_dataset["date"].min())
print("End:  ", ml_dataset["date"].max())

print("\nYear distribution:")
display(
    ml_dataset["date"]
    .dt.year
    .value_counts()
    .sort_index()
    .to_frame("rows")
)

Overall date range:
Start: 1964-06-19 00:00:00
End:   2026-09-10 00:00:00

Year distribution:


,rows
date,
1964,134
1965,159
1966,253
1967,272
1968,472
...,...
2022,211657
2023,219910
2024,226139


In [101]:
yearly_stats = (
    ml_dataset
    .assign(year=ml_dataset["date"].dt.year)
    .groupby("year")
    .agg(
        rows=("flood_next_day", "size"),
        flood_events=("flood_next_day", "sum")
    )
)

yearly_stats["flood_rate"] = (
    yearly_stats["flood_events"]
    / yearly_stats["rows"]
)

display(yearly_stats)

,rows,flood_events,flood_rate
year,,,
1964,134,0.0,0.000000
1965,159,0.0,0.000000
1966,253,0.0,0.000000
1967,272,4.0,0.014706
1968,472,1.0,0.002119
...,...,...,...
2022,211657,3128.0,0.014779
2023,219910,1724.0,0.007840
2024,226139,3049.0,0.013483


In [102]:
train_df = ml_dataset[
    ml_dataset["date"].dt.year <= 2022
].copy()

val_df = ml_dataset[
    ml_dataset["date"].dt.year.isin([2023, 2024])
].copy()

test_df = ml_dataset[
    ml_dataset["date"].dt.year >= 2025
].copy()

In [103]:
print("TRAIN")
print("Rows:", len(train_df))
print(
    "Date:",
    train_df["date"].min(),
    "→",
    train_df["date"].max()
)
print(
    "Flood:",
    int(train_df["flood_next_day"].sum())
)
print(
    "Flood rate:",
    train_df["flood_next_day"].mean()
)

print("\nVALIDATION")
print("Rows:", len(val_df))
print(
    "Date:",
    val_df["date"].min(),
    "→",
    val_df["date"].max()
)
print(
    "Flood:",
    int(val_df["flood_next_day"].sum())
)
print(
    "Flood rate:",
    val_df["flood_next_day"].mean()
)

print("\nTEST")
print("Rows:", len(test_df))
print(
    "Date:",
    test_df["date"].min(),
    "→",
    test_df["date"].max()
)
print(
    "Flood:",
    int(test_df["flood_next_day"].sum())
)
print(
    "Flood rate:",
    test_df["flood_next_day"].mean()
)

TRAIN
Rows: 2471787
Date: 1964-06-19 00:00:00 → 2022-12-31 00:00:00
Flood: 45862
Flood rate: 0.01855418771924927

VALIDATION
Rows: 446049
Date: 2023-01-01 00:00:00 → 2024-12-31 00:00:00
Flood: 4773
Flood rate: 0.01070061809352784

TEST
Rows: 374322
Date: 2025-01-01 00:00:00 → 2026-09-10 00:00:00
Flood: 4279
Flood rate: 0.011431334519477882


In [104]:
print(
    "Train/Validation overlap:",
    len(
        set(train_df["date"])
        & set(val_df["date"])
    )
)

print(
    "Validation/Test overlap:",
    len(
        set(val_df["date"])
        & set(test_df["date"])
    )
)

print(
    "Train/Test overlap:",
    len(
        set(train_df["date"])
        & set(test_df["date"])
    )
)

Train/Validation overlap: 0
Validation/Test overlap: 0
Train/Test overlap: 0


In [105]:
model_features = [
    "water_level",
    "water_level_lag_1",
    "water_level_lag_2",
    "water_level_lag_3",
    "level_change_1d",
    "level_change_3d",
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "level_danger_ratio",
    "level_warning_ratio",
    "rise_danger_ratio_1d",
    "rise_danger_ratio_3d",
    "danger_level",
    "warning_level",
    "lat",
    "lon"
]

In [106]:
X_train = train_df[model_features].copy()
y_train = train_df["flood_next_day"].astype(int)

X_val = val_df[model_features].copy()
y_val = val_df["flood_next_day"].astype(int)

X_test = test_df[model_features].copy()
y_test = test_df["flood_next_day"].astype(int)

In [107]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nValidation target:")
print(y_val.value_counts())

print("\nTest target:")
print(y_test.value_counts())

X_train: (2471787, 17)
y_train: (2471787,)
X_val: (446049, 17)
y_val: (446049,)
X_test: (374322, 17)
y_test: (374322,)

Training target:
flood_next_day
0    2425925
1      45862
Name: count, dtype: int64

Validation target:
flood_next_day
0    441276
1      4773
Name: count, dtype: int64

Test target:
flood_next_day
0    370043
1      4279
Name: count, dtype: int64


In [108]:
print("Number of features:", len(model_features))
print("\nFeatures:")
for i, feature in enumerate(model_features, start=1):
    print(f"{i:2}. {feature}")

Number of features: 17

Features:
 1. water_level
 2. water_level_lag_1
 3. water_level_lag_2
 4. water_level_lag_3
 5. level_change_1d
 6. level_change_3d
 7. rain_1d
 8. rain_3d
 9. rain_7d
10. level_danger_ratio
11. level_warning_ratio
12. rise_danger_ratio_1d
13. rise_danger_ratio_3d
14. danger_level
15. warning_level
16. lat
17. lon


In [109]:
print("\nX_train columns:")
print(X_train.columns.tolist())


X_train columns:
['water_level', 'water_level_lag_1', 'water_level_lag_2', 'water_level_lag_3', 'level_change_1d', 'level_change_3d', 'rain_1d', 'rain_3d', 'rain_7d', 'level_danger_ratio', 'level_warning_ratio', 'rise_danger_ratio_1d', 'rise_danger_ratio_3d', 'danger_level', 'warning_level', 'lat', 'lon']


In [110]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

print(logistic_model)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])


In [111]:
import time

start_time = time.time()

logistic_model.fit(X_train, y_train)

training_time = time.time() - start_time

print(f"Logistic Regression training completed.")
print(f"Training time: {training_time:.2f} seconds")

Logistic Regression training completed.
Training time: 11.64 seconds


In [112]:
logistic_estimator = logistic_model.named_steps["model"]

print("Converged iterations:", logistic_estimator.n_iter_)
print("Maximum iterations:", logistic_estimator.max_iter)
print("Number of coefficients:", logistic_estimator.coef_.shape)
print("Intercept:", logistic_estimator.intercept_)

Converged iterations: [80]
Maximum iterations: 1000
Number of coefficients: (1, 17)
Intercept: [-7.50920153]


In [113]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Predict probability of flood
val_prob = logistic_model.predict_proba(X_val)[:, 1]

# Default threshold = 0.5
val_pred = (val_prob >= 0.5).astype(int)

print("Validation ROC-AUC:", roc_auc_score(y_val, val_prob))
print("Validation PR-AUC:", average_precision_score(y_val, val_prob))
print("Validation Precision:", precision_score(y_val, val_pred, zero_division=0))
print("Validation Recall:", recall_score(y_val, val_pred, zero_division=0))
print("Validation F1:", f1_score(y_val, val_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred))

Validation ROC-AUC: 0.9872483123418783
Validation PR-AUC: 0.725539776104448
Validation Precision: 0.30533689126084057
Validation Recall: 0.9589356798659124
Validation F1: 0.4631887871274604

Confusion Matrix:
[[430863  10413]
 [   196   4577]]


In [114]:
from sklearn.metrics import precision_recall_curve
import numpy as np
import pandas as pd

precision, recall, thresholds = precision_recall_curve(
    y_val,
    val_prob
)

threshold_results = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1]
})

threshold_results["f1"] = (
    2 * threshold_results["precision"] * threshold_results["recall"]
    / (
        threshold_results["precision"]
        + threshold_results["recall"]
    )
)

# Show some useful threshold points
selected_thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

print(
    threshold_results[
        threshold_results["threshold"].isin(selected_thresholds)
    ].to_string(index=False)
)

Empty DataFrame
Columns: [threshold, precision, recall, f1]
Index: []


In [115]:
best_f1_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

print("Best validation F1 threshold:")
print(best_f1_row)

Best validation F1 threshold:
threshold    0.739413
precision    0.539363
recall       0.887073
f1           0.670839
Name: 401649, dtype: float64


In [116]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds_to_check = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.739413,
    0.80,
    0.90
]

rows = []

for threshold in thresholds_to_check:
    pred = (val_prob >= threshold).astype(int)

    rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "predicted_floods": pred.sum()
    })

threshold_table = pd.DataFrame(rows)

print(threshold_table.to_string(index=False))

 threshold  precision   recall       f1  predicted_floods
  0.300000   0.144421 0.978630 0.251697             32343
  0.400000   0.223832 0.974021 0.364014             20770
  0.500000   0.305337 0.958936 0.463189             14990
  0.600000   0.400909 0.942175 0.562477             11217
  0.700000   0.497874 0.907605 0.643016              8701
  0.739413   0.539304 0.886864 0.670734              7849
  0.800000   0.571358 0.726378 0.639609              6068
  0.900000   0.696390 0.602137 0.645843              4127


In [117]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

print(rf_model)

RandomForestClassifier(class_weight='balanced', max_depth=20,
                       min_samples_leaf=5, n_estimators=200, n_jobs=-1,
                       random_state=42)


In [118]:
import time

start_time = time.time()

rf_model.fit(X_train, y_train)

rf_training_time = time.time() - start_time

print("Random Forest training completed.")
print(f"Training time: {rf_training_time:.2f} seconds")

Random Forest training completed.
Training time: 206.34 seconds


In [119]:
import time

start_time = time.time()

rf_val_prob = rf_model.predict_proba(X_val)[:, 1]

rf_prediction_time = time.time() - start_time

print(f"Validation prediction time: {rf_prediction_time:.2f} seconds")
print(f"Probability shape: {rf_val_prob.shape}")
print(f"Minimum probability: {rf_val_prob.min():.6f}")
print(f"Maximum probability: {rf_val_prob.max():.6f}")

Validation prediction time: 0.80 seconds
Probability shape: (446049,)
Minimum probability: 0.000000
Maximum probability: 0.999961


In [120]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

rf_val_pred = (rf_val_prob >= 0.5).astype(int)

print("Random Forest — Validation Results")
print("----------------------------------")
print(f"ROC-AUC:  {roc_auc_score(y_val, rf_val_prob):.6f}")
print(f"PR-AUC:   {average_precision_score(y_val, rf_val_prob):.6f}")
print(f"Precision:{precision_score(y_val, rf_val_pred, zero_division=0):.6f}")
print(f"Recall:   {recall_score(y_val, rf_val_pred, zero_division=0):.6f}")
print(f"F1:       {f1_score(y_val, rf_val_pred, zero_division=0):.6f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, rf_val_pred))

Random Forest — Validation Results
----------------------------------
ROC-AUC:  0.994601
PR-AUC:   0.936750
Precision:0.736746
Recall:   0.931699
F1:       0.822833

Confusion Matrix:
[[439687   1589]
 [   326   4447]]


In [121]:
from sklearn.metrics import precision_recall_curve

rf_precision, rf_recall, rf_thresholds = precision_recall_curve(
    y_val,
    rf_val_prob
)

rf_threshold_results = pd.DataFrame({
    "threshold": rf_thresholds,
    "precision": rf_precision[:-1],
    "recall": rf_recall[:-1]
})

rf_threshold_results["f1"] = (
    2 * rf_threshold_results["precision"] * rf_threshold_results["recall"]
    / (
        rf_threshold_results["precision"]
        + rf_threshold_results["recall"]
    )
)

best_rf_f1 = rf_threshold_results.loc[
    rf_threshold_results["f1"].idxmax()
]

print("Best Random Forest validation F1 threshold:")
print(best_rf_f1)

Best Random Forest validation F1 threshold:
threshold    0.904376
precision    0.908869
recall       0.856694
f1           0.882010
Name: 301209, dtype: float64


In [122]:
thresholds_to_check = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

rows = []

for threshold in thresholds_to_check:
    pred = (rf_val_prob >= threshold).astype(int)

    rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "predicted_floods": pred.sum()
    })

rf_threshold_table = pd.DataFrame(rows)

print(rf_threshold_table.to_string(index=False))

 threshold  precision   recall       f1  predicted_floods
       0.3   0.624295 0.950765 0.753695              7269
       0.4   0.687500 0.942594 0.795087              6544
       0.5   0.736746 0.931699 0.822833              6036
       0.6   0.775388 0.921433 0.842125              5672
       0.7   0.818698 0.906348 0.860296              5284
       0.8   0.860801 0.887492 0.873943              4921
       0.9   0.905731 0.857532 0.880973              4519


In [123]:
scale_pos_weight = (
    y_train.value_counts()[0]
    / y_train.value_counts()[1]
)

print(f"Negative samples: {y_train.value_counts()[0]:,}")
print(f"Positive samples: {y_train.value_counts()[1]:,}")
print(f"scale_pos_weight: {scale_pos_weight:.6f}")

Negative samples: 2,425,925
Positive samples: 45,862
scale_pos_weight: 52.896189


In [124]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

print(xgb_model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)


In [125]:
import time

start_time = time.time()

xgb_model.fit(
    X_train,
    y_train
)

xgb_training_time = time.time() - start_time

print("XGBoost training completed.")
print(f"Training time: {xgb_training_time:.2f} seconds")

XGBoost training completed.
Training time: 42.55 seconds


In [126]:
import time

start_time = time.time()

xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]

xgb_prediction_time = time.time() - start_time

print(f"Validation prediction time: {xgb_prediction_time:.2f} seconds")
print(f"Probability shape: {xgb_val_prob.shape}")
print(f"Minimum probability: {xgb_val_prob.min():.6f}")
print(f"Maximum probability: {xgb_val_prob.max():.6f}")

Validation prediction time: 0.59 seconds
Probability shape: (446049,)
Minimum probability: 0.000007
Maximum probability: 0.999950


In [127]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

xgb_val_pred = (xgb_val_prob >= 0.5).astype(int)

print("XGBoost — Validation Results")
print("----------------------------")
print(f"ROC-AUC:  {roc_auc_score(y_val, xgb_val_prob):.6f}")
print(f"PR-AUC:   {average_precision_score(y_val, xgb_val_prob):.6f}")
print(f"Precision:{precision_score(y_val, xgb_val_pred, zero_division=0):.6f}")
print(f"Recall:   {recall_score(y_val, xgb_val_pred, zero_division=0):.6f}")
print(f"F1:       {f1_score(y_val, xgb_val_pred, zero_division=0):.6f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, xgb_val_pred))

XGBoost — Validation Results
----------------------------
ROC-AUC:  0.994869
PR-AUC:   0.938235
Precision:0.630311
Recall:   0.954117
F1:       0.759127

Confusion Matrix:
[[438605   2671]
 [   219   4554]]


In [128]:
xgb_precision, xgb_recall, xgb_thresholds = precision_recall_curve(
    y_val,
    xgb_val_prob
)

xgb_threshold_results = pd.DataFrame({
    "threshold": xgb_thresholds,
    "precision": xgb_precision[:-1],
    "recall": xgb_recall[:-1]
})

xgb_threshold_results["f1"] = (
    2 * xgb_threshold_results["precision"] * xgb_threshold_results["recall"]
    / (
        xgb_threshold_results["precision"]
        + xgb_threshold_results["recall"]
    )
)

best_xgb_f1 = xgb_threshold_results.loc[
    xgb_threshold_results["f1"].idxmax()
]

print("Best XGBoost validation F1 threshold:")
print(best_xgb_f1)

Best XGBoost validation F1 threshold:
threshold    0.955972
precision    0.890060
recall       0.866750
f1           0.878251
Name: 370864, dtype: float64


In [129]:
thresholds_to_check = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

rows = []

for threshold in thresholds_to_check:
    pred = (xgb_val_prob >= threshold).astype(int)

    rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "predicted_floods": pred.sum()
    })

xgb_threshold_table = pd.DataFrame(rows)

print(xgb_threshold_table.to_string(index=False))

 threshold  precision   recall       f1  predicted_floods
       0.3   0.523257 0.963964 0.678313              8793
       0.4   0.581723 0.958936 0.724152              7868
       0.5   0.630311 0.954117 0.759127              7225
       0.6   0.674332 0.946155 0.787446              6697
       0.7   0.720122 0.937984 0.814741              6217
       0.8   0.768776 0.926461 0.840285              5752
       0.9   0.834592 0.902787 0.867351              5163
